In [0]:
dbutils.widgets.text("source_table", "workspace.cdf_benchmark_source.bookings")
dbutils.widgets.text("target_table", "workspace.cdf_benchmark_structured_streaming_standard.bookings_current")
dbutils.widgets.text("checkpoint_path", "dbfs:/tmp/cdf_benchmark/structured_streaming_standard")
dbutils.widgets.text("starting_version", "0")

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
checkpoint_path = dbutils.widgets.get("checkpoint_path")
starting_version = dbutils.widgets.get("starting_version")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:target_table) (
  booking_id BIGINT,
  customer_id BIGINT,
  property_id BIGINT,
  booking_date DATE,
  checkin_date DATE,
  checkout_date DATE,
  amount DECIMAL(12,2),
  status STRING,
  updated_at TIMESTAMP
)
CLUSTER BY (booking_date)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def upsert_cdf(microbatch_df, batch_id):
    staged = (
        microbatch_df
        .filter(F.col("_change_type") != F.lit("update_preimage"))
        .withColumn(
            "_rn",
            F.row_number().over(
                Window.partitionBy("booking_id", "booking_date").orderBy(
                    F.col("updated_at").desc(),
                    F.col("_commit_version").desc(),
                    F.col("_commit_timestamp").desc(),
                )
            ),
        )
        .filter(F.col("_rn") == 1)
        .select(
            "booking_id",
            "customer_id",
            "property_id",
            "booking_date",
            "checkin_date",
            "checkout_date",
            "amount",
            "status",
            "updated_at",
            "_change_type",
        )
    )

    staged.createOrReplaceTempView("staged")

    spark.sql(f'''
    MERGE INTO {target_table} t
    USING staged s
    ON t.booking_date = s.booking_date AND t.booking_id = s.booking_id
    WHEN MATCHED AND s._change_type = 'delete' AND s.updated_at >= t.updated_at THEN DELETE
    WHEN MATCHED AND s._change_type IN ('insert', 'update_postimage') AND s.updated_at >= t.updated_at THEN UPDATE SET
      t.customer_id = s.customer_id,
      t.property_id = s.property_id,
      t.checkin_date = s.checkin_date,
      t.checkout_date = s.checkout_date,
      t.amount = s.amount,
      t.status = s.status,
      t.updated_at = s.updated_at
    WHEN NOT MATCHED AND s._change_type IN ('insert', 'update_postimage') THEN INSERT (
      booking_id,
      customer_id,
      property_id,
      booking_date,
      checkin_date,
      checkout_date,
      amount,
      status,
      updated_at
    )
    VALUES (
      s.booking_id,
      s.customer_id,
      s.property_id,
      s.booking_date,
      s.checkin_date,
      s.checkout_date,
      s.amount,
      s.status,
      s.updated_at
    )
    ''')

(
    spark.readStream
    .option("readChangeFeed", "true")
    .option("startingVersion", starting_version)
    .table(source_table)
    .writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .foreachBatch(upsert_cdf)
    .start()
    .awaitTermination()
)